In [12]:
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate


1. Download Ollama and run `.exe` file locally: https://ollama.com/download
2. Run the following command in your terminal to download a model:
```bash
ollama pull model_name
```
3. More info (here)[https://github.com/ollama/ollama]

### Test with `PyMuPDFLoader` and Ollama embeddings

In [ ]:
# load document from pdf with each page as a separate document
loader = PyMuPDFLoader(r"C:\Users\rpurohit\Documents\GitHub\llm-benchmarking\data\temp\new-haven-zoning-code.pdf")
documents = loader.load()

In [ ]:
# split documents into chunks but keep the content of each page as size of the chunk
text_splitter = RecursiveCharacterTextSplitter()
texts = text_splitter.split_documents(documents)

In [19]:
len(texts[5].page_content)

3913

In [ ]:
# select embedding model
emb_model = OllamaEmbeddings(model="nomic-embed-text")

# create vector store from documents
#vectore_store = FAISS.from_documents(documents=texts, embedding=emb_model)

In [22]:
# save vector store locally
vectore_store.save_local("../data/temp/new_haven_zoning_code_ollama_vector_db")

In [23]:
# load vector store
emb_vector_db = FAISS.load_local("../data/temp/new_haven_zoning_code_ollama_vector_db", 
                                emb_model,
                                allow_dangerous_deserialization=True
        )

In [24]:
# retrieve relevant context based on query

# Question based on Section 17. RO Districts: Residence-Office, Page 47 of the PDF
query = "For what purposes can a building be constructed in an RO District?"

retrieved_context = emb_vector_db.similarity_search(query, k=5)

In [32]:
retrieved_context[1].page_content

'Created: 2021-04-06 13:10:00 [EST] \n(Supp. No. 27) \n \nPage 154 of 211 \n• \nSelection of sites which lend themselves to visual mitigation.  \n• \nAvailability of road access.  \n• \nAvailability of electric power.  \n• \nAvailability of land based telephone lines or microwave link capability.  \nIf a tower is proposed the application shall include support materials that show the location of tall \nstructures within one quarter mile radius of the site proposed, that the owners of those locations have \nbeen contacted and asked for permission to install the antenna on those structures and denied for \nother than economic reasons. This would include smoke stacks, water towers, tall buildings, antennas \nor towers of other wireless communications companies, other communication towers (fire, police, etc.) \nand other tall structures.  \nThe City Plan Commission may deny an application to construct a newtowerif it is determined that the \napplicant has not made a good faith effort to mou

In [4]:
# load gemma3 model from ollama
model = ChatOllama(
    model="gemma3"
)

In [ ]:
# create prompt template for RAG
from langchain_core.output_parsers import StrOutputParser

RAG_TEMPLATE = """
You are a helpful assistant that has knowledge about the New Haven Zoning Code.
Based on the context provided, your task is to answer a user's question.
Only use the information provided in the context to answer the question. 
If the answer cannot be deduced from the context, then do not answer the questions.

<context>
{context}
</context>

Answer the following question:

{question}"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

chain = (rag_prompt
    | model
    | StrOutputParser()
)

question = "For what purposes can a building be constructed in an RO District?"

# Run
chain.invoke({"context": retrieved_context[0].page_content, "question": question})

### Test with previously created embeddings with `PyPDF` and a Hugging Face embedding model

In [7]:
# using previously saved vector store

# embedding model
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name= "intfloat/e5-small-v2",
    multi_process=True,
    encode_kwargs={"normalize_embeddings": True},  # set `True` for cosine similarity
)

prev_emb_vector_db = FAISS.load_local("../data/temp/new_haven_zoning_code_vector_db", 
                                embedding_model,
                                allow_dangerous_deserialization=True
        )

In [9]:
ret_context = prev_emb_vector_db.similarity_search(question, k=5)

In [10]:
ret_context[0].page_content

'offices, and to such other non-residential uses as generally support and harmonize with a high density area of this \ntype. The non-residential uses permitted in RO Districts, subject to adequate conditions and safeguards, are \nhereby found and declared to be the only appropriate such uses for such areas. It is hereby found and declared, \nfurther, that these regulations are necessary to the protection of these areas and that their protection is essential \nto the maintenance of a balanced community of sound residential areas of diverse types.  \nAll RO Districts are subject to the General Provisions for Residence Districts set forth in Article IV as well as to all \nother provisions of this ordinance.  \nUses permitted. In an RO District a building or other structure may be erected, altered, arranged, designed or \nused, and a lot or structure may be used for any of the following purposes and no other:'

In [11]:
chain.invoke({"context": ret_context[0].page_content, "question": question})

'A building or other structure may be erected, altered, arranged, designed or used for any of the following purposes: a building or other structure may be erected, altered, arranged, designed or used, and a lot or structure may be used for any of the following purposes and no other.'

### Test with smaller, focused PDF from original zoning code

In [ ]:
# load document from pdf with each page as a separate document
# page 40 to 60 of new haven zoning code pdf
loader = PyPDFLoader(r"C:\Users\rpurohit\Documents\GitHub\llm-benchmarking\data\temp\new-haven-zoning-code-select-pages.pdf")
documents = loader.load()

In [14]:
# split documents into chunks but keep the content of each page as size of the chunk
text_splitter = RecursiveCharacterTextSplitter()
texts = text_splitter.split_documents(documents)

In [16]:
slect_pages_vector =  FAISS.from_documents(documents=texts, embedding=embedding_model)

In [17]:
slect_pages_vector.save_local("../data/temp/new_haven_zoning_code_select_pages_vector_db")

# load vector store
select_pages_vector_db = FAISS.load_local("../data/temp/new_haven_zoning_code_select_pages_vector_db", 
                                embedding_model,
                                allow_dangerous_deserialization=True
        )

In [18]:
select_ret_context = select_pages_vector_db.similarity_search(question, k=5)

In [19]:
select_ret_context[0].page_content

'Created: 2021-04-06 13:09:56 [EST] \n(Supp. No. 27) \n \nPage 47 of 211 \nDepartment of Transportation, as follows: (i) uses listed in subsections 31(b)(1) through (10) and \n(13) (but not subject to the conditions of section 31), section 42C (but not including Package \nAlcoholic liquor), section 42D (but not including a Funeral home, gun and weapons repair, \nfirearms training, firing range, shop or a swap shop), section 42G, section 42H (but not including \ngun shops), section 42I and (ii) a Restaurant, caterer, music, or dancing school, The aggregate \ngross floor area of the foregoing uses shall not exceed 15% of the total gross floor areaof the \nbuilding in which they are located.  \nWhere both professional offices and retail uses are located in the same building, the combined \ngross floor area of the professional offices and retail uses shall be no greater than 15% of the \ntotal gross floor area of the building in which they are located. In addition, no parking spaces \nshal

In [20]:
chain.invoke({"context": select_ret_context[0].page_content, "question": question})

'A building in an RO District may be erected, altered, arranged, designed or used for:\n\n*   Residential uses as permitted in RH-1 Districts.\n*   Offices and studios of doctors, dentists, architects, artists, designers, accountants, lawyers, engineers, tutors, real estate and insurance agents, brokers, and members of other recognized professions.'